In [1]:
import pathlib
from pathlib import Path
import sys
import pyarrow.parquet as pq
from DataFile.data_name import DataName
from typing import List

In [3]:
import dask.dataframe as dd
from pathlib import Path

# 获取 Parquet 文件列表
pfs = list(Path(r"E:\tmp\OKX-Books-BTC-USDT-400").glob("*.parquet"))

# 使用 Dask 读取多个 Parquet 文件
ddf = dd.read_parquet(pfs, columns=["action","ts"])

# 将每个分区与文件路径配对，并转换为字典
actions = {}
for i, pf in enumerate(pfs):
    # 获取对应分区的 action 列并计算
    partition = ddf.partitions[i]
    actions[pf] = partition["action"].compute().to_list()

In [4]:
stat = {}
for pf, action_list in actions.items():
    snapshot_count = action_list.count("snapshot")
    update_count = action_list.count("update")
    stat[pf] = {
        "snapshot_count": snapshot_count,
        "update_count": update_count,
        "total_count": len(action_list)
    }

In [5]:
# to csv
import pandas as pd
df_stat = pd.DataFrame.from_dict(stat, orient='index')
df_stat.to_csv("action_statistics.csv")

In [6]:
total_snapshot_count = sum(item['snapshot_count'] for item in stat.values())
print(f"Total snapshot count: {total_snapshot_count}")

Total snapshot count: 338


In [7]:
len(pfs)

51

In [13]:
p = pq.ParquetFile(pfs[0])
print(p.metadata)

  created_by: parquet-cpp-arrow version 19.0.0
  num_columns: 10
  num_rows: 1000000
  num_row_groups: 1
  format_version: 2.6
  serialized_size: 2580


1000000